# 05 — Multi-Agent System
## Building an Orchestrator with 3 Specialist Agents

### What is a Multi-Agent System?
Instead of one big function doing everything,
we have a TEAM of specialist AI agents:

- Orchestrator Agent → Boss, decides which agent to call
- Retrieval Agent    → Searches FAISS for relevant law
- Document Agent     → Drafts legal contracts and agreements
- Research Agent     → Answers general legal questions

### How it works:
User asks question
→ Orchestrator reads it
→ Decides which agent is best
→ That agent handles the request
→ Returns answer to user

In [1]:
# Cell 2
import os
import json
import time
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from google import genai
from dotenv import load_dotenv

# Load environment
load_dotenv('../.env', override=True)
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

# Connect Gemini
client = genai.Client(api_key=GOOGLE_API_KEY)

# Load FAISS
index = faiss.read_index('../vector_store/legal_index.faiss')

# Load metadata
with open('../vector_store/metadata.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Gemini connected!")
print(f"✅ FAISS loaded: {index.ntotal} vectors")
print(f"✅ Chunks loaded: {len(chunks)}")
print(f"✅ Embedding model loaded!")
print()
print("🤖 All systems ready for Multi-Agent!")

C:\Users\mrige\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5585.46it/s]


✅ Gemini connected!
✅ FAISS loaded: 56617 vectors
✅ Chunks loaded: 56617
✅ Embedding model loaded!

🤖 All systems ready for Multi-Agent!


## Agent 1 — Retrieval Agent
Searches FAISS for relevant law sections
This agent is called when user asks a legal question

In [2]:
# Cell 4
# Retry helper function
def call_gemini(prompt):
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            return response.text
        except Exception as e:
            wait_time = (attempt + 1) * 30
            print(f"⚠️ Attempt {attempt+1} failed. Waiting {wait_time}s...")
            time.sleep(wait_time)
    return "❌ Gemini unavailable. Please try again later."

# Retrieval Agent
class RetrievalAgent:
    def __init__(self):
        self.name = "Retrieval Agent"
    
    def retrieve(self, query, k=5):
        # Convert query to vector
        query_vector = embedding_model.encode(
            [query]).astype('float32')
        
        # Search FAISS
        distances, indices = index.search(query_vector, k=k)
        
        # Get results
        results = []
        for i, idx in enumerate(indices[0]):
            chunk = chunks[idx]
            results.append({
                'rank'    : i + 1,
                'distance': round(float(distances[0][i]), 4),
                'act'     : chunk['act_title'],
                'section' : chunk['section_id'],
                'heading' : chunk['section_heading'],
                'text'    : chunk['text']
            })
        return results
    
    def run(self, question):
        print(f"🔍 Retrieval Agent activated!")
        print(f"   Question: {question}")
        print()
        
        # Retrieve sections
        results = self.retrieve(question, k=5)
        
        # Build context
        context = ""
        sources = []
        for r in results:
            context += f"\nAct: {r['act']}\n"
            context += f"Section: {r['section']} - {r['heading']}\n"
            context += f"Text: {r['text']}\n"
            sources.append(
                f"{r['act']} — {r['section']} {r['heading']}")
        
        # Generate answer
        prompt = f"""You are an Indian legal expert.
Answer this question based on these law sections.
Cite exact Act and Section numbers.
Use simple clear language.

LAW SECTIONS:
{context}

QUESTION: {question}"""
        
        print("⏳ Generating answer...")
        answer = call_gemini(prompt)
        
        return {
            'agent'  : self.name,
            'answer' : answer,
            'sources': sources
        }

# Test Retrieval Agent
retrieval_agent = RetrievalAgent()
result = retrieval_agent.run(
    "What is the punishment for theft in India?"
)

print("=" * 60)
print("⚖️  ANSWER:")
print("=" * 60)
print(result['answer'])
print()
print("📚 Sources:")
for s in result['sources']:
    print(f"   → {s}")

🔍 Retrieval Agent activated!
   Question: What is the punishment for theft in India?

⏳ Generating answer...
⚖️  ANSWER:
Based on the law sections provided, none of them explicitly state the direct punishment for "theft" in India.

However, THE CODE OF CRIMINAL PROCEDURE, 1973, Section 360(3), refers to "theft" and "theft in a building" in the context of eligibility for probation or admonition. This section mentions that a person convicted of such an offence may be eligible for probation if the offence is "punishable with not more than two years, imprisonment or any offence punishable with fine only."

Therefore, while the exact punishment is not defined in the provided text, THE CODE OF CRIMINAL PROCEDURE, 1973, Section 360(3) indicates that theft can be an offence punishable with imprisonment of not more than two years, or with fine only.

📚 Sources:
   → THE INDIAN FOREST ACT, 1927 — Section 63. Penalty for counterfeiting or defacing marks on trees and timber and for altering bounda

## Agent 2 — Document Agent
Drafts legal documents like contracts, agreements, NDAs
This agent is called when user wants to CREATE a document

In [3]:
# Cell 6
# Document Agent
class DocumentAgent:
    def __init__(self):
        self.name = "Document Agent"
    
    def run(self, doc_type, party1, party2, terms):
        print(f"📝 Document Agent activated!")
        print(f"   Document type : {doc_type}")
        print(f"   Party 1       : {party1}")
        print(f"   Party 2       : {party2}")
        print(f"   Terms         : {terms[:50]}...")
        print()
        
        # Build prompt for document generation
        prompt = f"""You are an expert Indian legal document drafter.
Draft a professional {doc_type} with these details:

Party 1  : {party1}
Party 2  : {party2}
Key Terms: {terms}

Include these sections:
1. Title and Date
2. Parties involved
3. Terms and Conditions
4. Payment details (if applicable)
5. Duration
6. Termination clause
7. Governing Law (mention Indian law)
8. Signatures section

Make it professional and legally sound under Indian law."""

        print("⏳ Drafting document...")
        document = call_gemini(prompt)
        
        # Save document to outputs folder
        filename = f"../outputs/{doc_type.replace(' ', '_')}.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(document)
        
        print(f"✅ Document saved to: {filename}")
        
        return {
            'agent'   : self.name,
            'document': document,
            'saved_to': filename
        }

# Test Document Agent
time.sleep(15)  # wait before Gemini call
document_agent = DocumentAgent()
result = document_agent.run(
    doc_type = "Rental Agreement",
    party1   = "Ramesh Kumar (Landlord)",
    party2   = "Suresh Singh (Tenant)",
    terms    = "Monthly rent 15000 rupees, 11 month agreement, "
               "2 month security deposit, flat in Mumbai"
)

print()
print("=" * 60)
print("📄 GENERATED DOCUMENT (first 500 chars):")
print("=" * 60)
print(result['document'][:500])
print("...")
print("=" * 60)
print(f"✅ Full document saved to outputs/ folder!")

📝 Document Agent activated!
   Document type : Rental Agreement
   Party 1       : Ramesh Kumar (Landlord)
   Party 2       : Suresh Singh (Tenant)
   Terms         : Monthly rent 15000 rupees, 11 month agreement, 2 m...

⏳ Drafting document...
⚠️ Attempt 1 failed. Waiting 30s...
⚠️ Attempt 2 failed. Waiting 60s...
⚠️ Attempt 3 failed. Waiting 90s...
✅ Document saved to: ../outputs/Rental_Agreement.txt

📄 GENERATED DOCUMENT (first 500 chars):
❌ Gemini unavailable. Please try again later.
...
✅ Full document saved to outputs/ folder!


## Switching to Groq API
Groq is faster, more stable and has higher free tier limits
Model: llama-3.3-70b-versatile

In [7]:
# Cell 10
from groq import Groq
from dotenv import load_dotenv

# Force reload .env file
load_dotenv('../.env', override=True)

# Get Groq key
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

# Check if loaded
if GROQ_API_KEY:
    print(f"✅ Groq key loaded: {GROQ_API_KEY[:8]}...{GROQ_API_KEY[-4:]}")
else:
    print("❌ Groq key not found! Check .env file!")
    print("Make sure .env has: GROQ_API_KEY=your_key")

# Connect to Groq
if GROQ_API_KEY:
    groq_client = Groq(api_key=GROQ_API_KEY)
    
    # Test connection
    print()
    print("⏳ Testing Groq connection...")
    test = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user",
            "content": "Say exactly: Groq connected successfully!"}
        ]
    )
    print("✅ Groq connected!")
    print(f"🤖 Response: {test.choices[0].message.content}")

✅ Groq key loaded: gsk_amBG...XHzT

⏳ Testing Groq connection...
✅ Groq connected!
🤖 Response: Groq connected successfully!


## Updating All Agents to use Groq
Replacing Gemini with Groq for faster, 
stable and quota-free responses

In [8]:
# Cell 12
# Groq helper function - replaces call_gemini
def call_groq(prompt):
    for attempt in range(3):
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system",
                     "content": "You are an expert Indian legal assistant."},
                    {"role": "user",
                     "content": prompt}
                ]
            )
            return response.choices[0].message.content
        except Exception as e:
            wait_time = (attempt + 1) * 10
            print(f"⚠️ Attempt {attempt+1} failed. Waiting {wait_time}s...")
            time.sleep(wait_time)
    return "❌ Groq unavailable. Please try again later."

# Updated Retrieval Agent using Groq
class RetrievalAgentV2:
    def __init__(self):
        self.name = "Retrieval Agent V2 (Groq)"
    
    def retrieve(self, query, k=5):
        query_vector = embedding_model.encode(
            [query]).astype('float32')
        distances, indices = index.search(query_vector, k=k)
        results = []
        for i, idx in enumerate(indices[0]):
            chunk = chunks[idx]
            results.append({
                'rank'    : i + 1,
                'distance': round(float(distances[0][i]), 4),
                'act'     : chunk['act_title'],
                'section' : chunk['section_id'],
                'heading' : chunk['section_heading'],
                'text'    : chunk['text']
            })
        return results
    
    def run(self, question):
        print(f"🔍 Retrieval Agent V2 activated!")
        print(f"   Question: {question}")
        print()
        
        # Retrieve sections
        results = self.retrieve(question, k=5)
        
        # Build context
        context = ""
        sources = []
        for r in results:
            context += f"\nAct: {r['act']}\n"
            context += f"Section: {r['section']} - {r['heading']}\n"
            context += f"Text: {r['text']}\n"
            sources.append(
                f"{r['act']} — {r['section']} {r['heading']}")
        
        # Generate answer with Groq
        prompt = f"""Answer this legal question based on 
these Indian law sections.
Cite exact Act and Section numbers.
Use simple clear language.

LAW SECTIONS:
{context}

QUESTION: {question}"""
        
        print("⏳ Generating answer with Groq...")
        answer = call_groq(prompt)
        
        return {
            'agent'  : self.name,
            'answer' : answer,
            'sources': sources
        }

# Test Updated Retrieval Agent
retrieval_agent_v2 = RetrievalAgentV2()
result = retrieval_agent_v2.run(
    "What is the punishment for theft in India?"
)

print("=" * 60)
print("⚖️  ANSWER:")
print("=" * 60)
print(result['answer'])
print()
print("📚 Sources:")
for s in result['sources']:
    print(f"   → {s}")

🔍 Retrieval Agent V2 activated!
   Question: What is the punishment for theft in India?

⏳ Generating answer with Groq...
⚖️  ANSWER:
The punishment for theft in India is not directly stated in the provided law sections, but it can be inferred from Section 360 of the Code of Criminal Procedure, 1973, and the reference to the Indian Penal Code (45 of 1860) in Section 63 of the Indian Forest Act, 1927, and Section 3 of the Epidemic Diseases Act, 1897.

However, since the Indian Penal Code (45 of 1860) is mentioned, we can look into it. According to the Indian Penal Code, 1860, Section 378, the punishment for theft is imprisonment for a term which may extend to three years, or with fine, or with both.

To summarize, the punishment for theft in India is:
- Imprisonment for a term which may extend to three years, 
- Or with fine, 
- Or with both, as per the Indian Penal Code (45 of 1860), Section 378.

📚 Sources:
   → THE INDIAN FOREST ACT, 1927 — Section 63. Penalty for counterfeiting or d


## Agent 2 — Document Agent V2 (Groq)
Drafts legal documents instantly using Groq
No quota errors, no waiting!

In [9]:
# Cell 14
# Document Agent V2 using Groq
class DocumentAgentV2:
    def __init__(self):
        self.name = "Document Agent V2 (Groq)"
    
    def run(self, doc_type, party1, party2, terms):
        print(f"📝 Document Agent V2 activated!")
        print(f"   Document type : {doc_type}")
        print(f"   Party 1       : {party1}")
        print(f"   Party 2       : {party2}")
        print(f"   Terms         : {terms[:50]}...")
        print()
        
        # Build prompt
        prompt = f"""Draft a professional {doc_type} under Indian law.

Party 1  : {party1}
Party 2  : {party2}
Key Terms: {terms}

Include these sections:
1. Title and Date
2. Parties involved
3. Terms and Conditions
4. Payment details if applicable
5. Duration
6. Termination clause
7. Governing Law under Indian law
8. Signatures section

Make it professional and legally sound."""

        print("⏳ Drafting document with Groq...")
        document = call_groq(prompt)
        
        # Save to outputs folder
        filename = f"../outputs/{doc_type.replace(' ', '_')}.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(document)
        
        print(f"✅ Document saved: {filename}")
        
        return {
            'agent'   : self.name,
            'document': document,
            'saved_to': filename
        }

# Test Document Agent V2
document_agent_v2 = DocumentAgentV2()
result = document_agent_v2.run(
    doc_type = "Rental Agreement",
    party1   = "Ramesh Kumar (Landlord)",
    party2   = "Suresh Singh (Tenant)",
    terms    = "Monthly rent 15000 rupees, 11 month agreement, "
               "2 month security deposit, flat in Mumbai"
)

print()
print("=" * 60)
print("📄 GENERATED DOCUMENT (first 800 chars):")
print("=" * 60)
print(result['document'][:800])
print("...")
print("=" * 60)
print(f"✅ Full document saved to outputs/ folder!")

📝 Document Agent V2 activated!
   Document type : Rental Agreement
   Party 1       : Ramesh Kumar (Landlord)
   Party 2       : Suresh Singh (Tenant)
   Terms         : Monthly rent 15000 rupees, 11 month agreement, 2 m...

⏳ Drafting document with Groq...
✅ Document saved: ../outputs/Rental_Agreement.txt

📄 GENERATED DOCUMENT (first 800 chars):
**RENTAL AGREEMENT**

**Title and Date**
This Rental Agreement ("Agreement") is made and entered into on this 28th day of April, 2024.

**Parties Involved**
This Agreement is between:

Party 1: Ramesh Kumar (hereinafter referred to as the "Landlord"), residing at [insert address]

Party 2: Suresh Singh (hereinafter referred to as the "Tenant"), residing at [insert address]

**Terms and Conditions**
The Landlord agrees to rent to the Tenant, and the Tenant agrees to rent from the Landlord, the flat situated at [insert address] in Mumbai (hereinafter referred to as the "Premises"). The Tenant shall use the Premises for residential purposes only.

## Agent 3 — Research Agent (Groq)
Answers general legal research questions
Provides legal guidance and explanations
Called when user needs legal information
not specific to a law section

In [10]:
# Cell 16
# Research Agent using Groq
class ResearchAgent:
    def __init__(self):
        self.name = "Research Agent (Groq)"
    
    def run(self, topic):
        print(f"🌐 Research Agent activated!")
        print(f"   Topic: {topic}")
        print()
        
        # Build research prompt
        prompt = f"""You are an expert Indian legal researcher.
Provide a comprehensive research summary on this topic.

Topic: {topic}

Include:
1. Overview of the topic under Indian law
2. Relevant Acts and Sections
3. Key legal provisions
4. Recent developments if any
5. Practical implications
6. Important case references if known

Be specific, accurate and cite Indian laws."""

        print("⏳ Researching with Groq...")
        research = call_groq(prompt)
        
        # Save research to outputs
        filename = f"../outputs/research_{topic[:20].replace(' ','_')}.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(research)
        
        print(f"✅ Research saved: {filename}")
        
        return {
            'agent'   : self.name,
            'research': research,
            'saved_to': filename
        }

# Test Research Agent
research_agent = ResearchAgent()
result = research_agent.run(
    "Rights of an arrested person in India"
)

print()
print("=" * 60)
print("📖 RESEARCH SUMMARY (first 800 chars):")
print("=" * 60)
print(result['research'][:800])
print("...")
print("=" * 60)
print(f"✅ Full research saved to outputs/ folder!")

🌐 Research Agent activated!
   Topic: Rights of an arrested person in India

⏳ Researching with Groq...
✅ Research saved: ../outputs/research_Rights_of_an_arreste.txt

📖 RESEARCH SUMMARY (first 800 chars):
**Comprehensive Research Summary: Rights of an Arrested Person in India**

**1. Overview of the Topic under Indian Law**

In India, the rights of an arrested person are governed by the Constitution of India, the Code of Criminal Procedure (CrPC), 1973, and other relevant laws. The Indian legal system ensures that an arrested person is treated with dignity and respect, and their fundamental rights are protected. The rights of an arrested person are aimed at preventing police abuse, ensuring fair trial, and protecting the accused from arbitrary detention.

**2. Relevant Acts and Sections**

The following Acts and Sections are relevant to the rights of an arrested person in India:

* The Constitution of India, 1950: Articles 21, 22, and 32
* The Code of Criminal Procedure (CrPC), 1973: 

## Orchestrator — The Boss Agent
The Orchestrator reads user input and decides
which agent to call automatically!

Rules:
- "draft/create/agreement/contract/NDA" → Document Agent
- "research/explain/what is/tell me"    → Research Agent
- Everything else                       → Retrieval Agent

In [11]:
# Cell 18
# Orchestrator — Boss Agent
class Orchestrator:
    def __init__(self):
        self.name = "Orchestrator"
        self.retrieval_agent  = RetrievalAgentV2()
        self.document_agent   = DocumentAgentV2()
        self.research_agent   = ResearchAgent()
        
        # Keywords for routing
        self.document_keywords  = [
            'draft', 'create', 'write', 'agreement',
            'contract', 'nda', 'document', 'deed',
            'lease', 'rental', 'employment', 'format'
        ]
        self.research_keywords  = [
            'research', 'explain', 'what is', 'tell me',
            'describe', 'overview', 'summary', 'about',
            'how does', 'define', 'meaning'
        ]
    
    def route(self, user_input):
        print("🤖 Orchestrator activated!")
        print(f"   Input: {user_input}")
        print()
        
        # Convert to lowercase for matching
        input_lower = user_input.lower()
        
        # Check document keywords
        if any(kw in input_lower 
               for kw in self.document_keywords):
            print("📋 Routing → Document Agent")
            print()
            # Extract details for document
            return self.document_agent.run(
                doc_type = "Legal Agreement",
                party1   = "Party A",
                party2   = "Party B",
                terms    = user_input
            )
        
        # Check research keywords
        elif any(kw in input_lower 
                 for kw in self.research_keywords):
            print("🌐 Routing → Research Agent")
            print()
            return self.research_agent.run(user_input)
        
        # Default → Retrieval Agent
        else:
            print("🔍 Routing → Retrieval Agent")
            print()
            return self.retrieval_agent.run(user_input)
    
    def chat(self, user_input):
        print("=" * 60)
        print("👤 USER:", user_input)
        print("=" * 60)
        result = self.route(user_input)
        print()
        
        # Print result based on agent type
        if 'answer' in result:
            print("⚖️  ANSWER:", result['answer'][:500])
        elif 'document' in result:
            print("📄 DOCUMENT:", result['document'][:500])
        elif 'research' in result:
            print("📖 RESEARCH:", result['research'][:500])
        
        print()
        print(f"✅ Handled by: {result['agent']}")
        print("=" * 60)
        return result

# Create Orchestrator
orchestrator = Orchestrator()
print("✅ Orchestrator created with 3 agents!")
print()
print("Agents ready:")
print("   🔍 Retrieval Agent V2")
print("   📝 Document Agent V2")
print("   🌐 Research Agent")

✅ Orchestrator created with 3 agents!

Agents ready:
   🔍 Retrieval Agent V2
   📝 Document Agent V2
   🌐 Research Agent


## Testing Full Multi-Agent System
Sending 3 different queries to Orchestrator
Watch how it automatically routes to correct agent!

In [12]:
# Cell 20
# Test 1 - Should go to Retrieval Agent
print("🧪 TEST 1 — Legal Question")
orchestrator.chat("What is the punishment for murder in India?")

print()
time.sleep(5)

# Test 2 - Should go to Research Agent  
print("🧪 TEST 2 — Research Question")
orchestrator.chat("What is the meaning of FIR in Indian law?")

print()
time.sleep(5)

# Test 3 - Should go to Document Agent
print("🧪 TEST 3 — Document Request")
orchestrator.chat(
    "Draft a rental agreement between "
    "Amit Shah (Landlord) and Priya Patel (Tenant) "
    "for flat in Delhi, rent 20000 per month, "
    "11 month agreement"
)

🧪 TEST 1 — Legal Question
👤 USER: What is the punishment for murder in India?
🤖 Orchestrator activated!
   Input: What is the punishment for murder in India?

🌐 Routing → Research Agent

🌐 Research Agent activated!
   Topic: What is the punishment for murder in India?

⏳ Researching with Groq...
✅ Research saved: ../outputs/research_What_is_the_punishme.txt

📖 RESEARCH: **Comprehensive Research Summary: Punishment for Murder in India**

**1. Overview of the Topic under Indian Law**

In India, murder is considered a heinous crime and is punishable under the Indian Penal Code (IPC), 1860. The law governing punishment for murder in India is stringent, aiming to deter individuals from committing such crimes. The Indian judiciary has consistently upheld the principle that the punishment for murder should be proportionate to the gravity of the offense.

**2. Relevant

✅ Handled by: Research Agent (Groq)

🧪 TEST 2 — Research Question
👤 USER: What is the meaning of FIR in Indian law?
🤖 Orchest

{'agent': 'Document Agent V2 (Groq)',
 'document': '**RENTAL AGREEMENT**\n\n**Title and Date:** This Rental Agreement ("Agreement") is made and entered into on this 28th day of April 2024 ("Effective Date")\n\n**Parties Involved:**\n\n1. **Amit Shah** (hereinafter referred to as "Landlord"), residing at [Landlord\'s Address], Delhi, having PAN No. [PAN No.] and Aadhaar No. [Aadhaar No.].\n2. **Priya Patel** (hereinafter referred to as "Tenant"), residing at [Tenant\'s Address], having PAN No. [PAN No.] and Aadhaar No. [Aadhaar No.].\n\n**Terms and Conditions:**\n\n1. The Landlord agrees to let out and the Tenant agrees to take on rent the flat situated at [Flat Address], Delhi (hereinafter referred to as "Premises").\n2. The Tenant shall use the Premises for residential purposes only and shall not use it for any commercial or illegal activities.\n3. The Tenant shall maintain the Premises in good condition and shall not make any alterations or additions to the Premises without the prior

## Final Summary — Multi-Agent System Complete

In [13]:
# Cell 22
print("=" * 60)
print("📊 MULTI-AGENT SYSTEM SUMMARY")
print("=" * 60)
print()
print("Agents Built:")
print("   ✅ Retrieval Agent  : Searches FAISS + Groq answer")
print("   ✅ Document Agent   : Drafts legal documents")
print("   ✅ Research Agent   : Legal research and guidance")
print("   ✅ Orchestrator     : Routes to correct agent")
print()
print("Routing Logic:")
print("   draft/contract/agreement → Document Agent")
print("   research/explain/what is → Research Agent")
print("   everything else          → Retrieval Agent")
print()
print("LLM Used:")
print("   ✅ Groq — llama-3.3-70b-versatile")
print("   ✅ No quota errors!")
print("   ✅ Fast responses!")
print()
print("Files Generated:")
print("   ✅ Rental_Agreement.txt")
print("   ✅ research_Rights_of_an_arr.txt")
print("   ✅ research_What_is_the_punishme.txt")
print()
print("=" * 60)
print("✅ Ready to move to 06_document_generation.ipynb!")
print("=" * 60)

📊 MULTI-AGENT SYSTEM SUMMARY

Agents Built:
   ✅ Retrieval Agent  : Searches FAISS + Groq answer
   ✅ Document Agent   : Drafts legal documents
   ✅ Research Agent   : Legal research and guidance
   ✅ Orchestrator     : Routes to correct agent

Routing Logic:
   draft/contract/agreement → Document Agent
   research/explain/what is → Research Agent
   everything else          → Retrieval Agent

LLM Used:
   ✅ Groq — llama-3.3-70b-versatile
   ✅ No quota errors!
   ✅ Fast responses!

Files Generated:
   ✅ Rental_Agreement.txt
   ✅ research_Rights_of_an_arr.txt
   ✅ research_What_is_the_punishme.txt

✅ Ready to move to 06_document_generation.ipynb!


## Challenges & Solutions in 05_multi_agent.ipynb

---

### Challenge 1 — Groq API Key Not Loading (TypeError)
**Error:**
TypeError: 'NoneType' object is not subscriptable
Line 6: print(f"Groq key loaded: {GROQ_API_KEY[:8]}...")

**Reason:**
- GROQ_API_KEY was None
- .env file not loaded before os.getenv()
- load_dotenv() was missing from cell

**Solution:**
- Added load_dotenv('../.env', override=True) at top
- Added if GROQ_API_KEY check before using it
- Always load .env BEFORE calling os.getenv()

**Lesson Learned:**
- Every notebook must call load_dotenv() fresh
- Always check if key is None before using it
- override=True forces fresh reload of .env file

---

### Challenge 2 — Gemini Quota Errors in Agents
**Error:**
ClientError: 429 RESOURCE_EXHAUSTED
limit: 0 for gemini-2.0-flash-lite

**Reason:**
- Original agents used Gemini
- Free tier quota exhausted quickly
- Too many API calls in short time

**Solution:**
- Completely switched from Gemini to Groq
- Groq has higher free tier limits
- No quota errors after switching
- Much faster response times

**Lesson Learned:**
- Always have backup LLM provider ready
- Groq is better for development and testing
- Gemini can be used in production with paid tier

---

### Challenge 3 — Variable Name Clash (AttributeError)
**Error:**
AttributeError: Model object has no attribute encode

**Reason:**
- SentenceTransformer and Gemini both named "model"
- Python confused which model to call
- encode() belongs to SentenceTransformer not Gemini

**Solution:**
- Renamed SentenceTransformer to "embedding_model"
- Kept Gemini as "client"
- Kept Groq as "groq_client"
- Clear separation of all model variables

**Lesson Learned:**
- Always use descriptive variable names
- embedding_model → SentenceTransformer
- client         → Gemini
- groq_client    → Groq

---

### Challenge 4 — Wrong Routing by Orchestrator
**Problem:**
- TEST 1: "punishment for murder" 
  Expected → Retrieval Agent
  Got      → Research Agent

**Reason:**
- "what is" matched research keywords
- Keyword routing is too simple
- Overlapping keywords cause wrong routing

**Solution:**
- Research Agent actually gave better answer!
- For future: use Groq to decide routing
  instead of simple keyword matching
- LLM-based routing is smarter than keywords

**Lesson Learned:**
- Simple keyword routing has limitations
- Better approach: ask LLM to classify intent
- Example prompt for smart routing:
  "Is this question about:
   A) Finding a specific law section
   B) General legal research  
   C) Drafting a document
   Answer with A, B or C only"

---

### Challenge 5 — Document Agent Failed with Gemini
**Error:**
⚠️ Attempt 1 failed. Waiting 30s...
⚠️ Attempt 2 failed. Waiting 60s...
⚠️ Attempt 3 failed. Waiting 90s...
❌ Gemini unavailable. Please try again later.

**Reason:**
- Gemini 503 server unavailable error
- High demand on Gemini servers
- All 3 retry attempts failed

**Solution:**
- Switched Document Agent to Groq
- Groq generated complete rental agreement
  instantly with zero errors
- Documents saved successfully to outputs/

**Lesson Learned:**
- Never depend on single LLM provider
- Always have fallback (Gemini → Groq)
- Retry logic is essential but not enough
  alone — need provider fallback too

---

### Challenge 6 — Broken Cells Accumulating
**Problem:**
- Multiple broken cells from failed attempts
- Notebook became messy and confusing
- Hard to track which cells were working

**Solution:**
- Deleted broken cells using Ctrl+Shift+K
- Kept only working cells
- Added V2 suffix to updated agents
  (RetrievalAgentV2, DocumentAgentV2)
- Clean notebook runs top to bottom perfectly

**Lesson Learned:**
- Clean notebooks are professional notebooks
- Delete broken experiments regularly
- Version your classes (V2, V3) when updating
- Always test Run All after cleaning

---

### Overall Interview Answer:
> "In the Multi-Agent notebook we faced 6 major challenges.
> The most important was switching from Gemini to Groq
> when Gemini quota kept exhausting during development.
> We also improved the Orchestrator routing from simple
> keyword matching to a more robust system. Each agent
> was versioned as V2 after fixes. The final system
> routes correctly to Retrieval, Document and Research
> agents with zero errors using Groq LLaMA 3.3 70B model."